# P-027 — E6 end-to-end training sweep (audit experiment, Colab T4)

This runs the experiment an external verification audit demanded for
**[The Dissipation Budget Law](https://srivtx.github.io/p-rick/paper/p027.html)** (p-rick working paper P-027, rev 1.1):
*train the port-Hamiltonian stream end-to-end* — against trained ResNet,
ResNet+LayerNorm, and orthogonal Cayley baselines — instead of only scoring
random streams.

**96 runs total** (4 architectures x 2 tasks x 3 seeds x 2 learning rates,
validation-selected). The harness is **seed-frozen and resumable**: it fetches
the runs already completed and committed in the repo and only runs the rest,
saving after every single run.

**How to use this notebook**
1. `Runtime -> Change runtime type -> T4 GPU -> Save` (the notebook asks for
   GPU automatically, but confirm it)
2. `Runtime -> Run all` (or Ctrl+F9)
3. Wait for the long cell (roughly **45–90 minutes** depending on how many
   runs the repo already banked — each job prints a line as it finishes)
4. The last cell prints the full `training-results.json` **between two
   marker lines** and auto-downloads the two output files. Copy the payload
   (or upload the downloaded file) back to the assistant, who writes the
   numbers into the paper, regenerates Figure 7, rebuilds the PDF, and ships.

Interrupting is safe: every completed run is saved; re-running the notebook
picks up where it left off.


In [ ]:
import torch

print('torch', torch.__version__)
gpu = torch.cuda.is_available()
print('CUDA available:', gpu)
if gpu:
    print('GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
else:
    print('NO GPU runtime — the sweep still runs on CPU, just slower.')
    print('Fix: Runtime -> Change runtime type -> T4 GPU -> Save, then')
    print('Runtime -> Run all again.')


In [ ]:
import json, os, urllib.request

os.makedirs('p-rick/code', exist_ok=True)
os.makedirs('p-rick/figures/p-027', exist_ok=True)
BASE = 'https://raw.githubusercontent.com/srivtx/p-rick/main/'
RES = 'p-rick/figures/p-027/training-results.json'

# the harness (seed-frozen, resumable)
urllib.request.urlretrieve(BASE + 'code/p-027b-training.py',
                           'p-rick/code/p-027b-training.py')
print('fetched code/p-027b-training.py')

# runs already completed and committed in the repo (resume state)
try:
    urllib.request.urlretrieve(BASE + 'figures/p-027/training-results.json',
                               RES)
    done = len(json.load(open(RES))['runs'])
    print(f'resumed from {done} completed runs already on the repo')
except Exception:
    if os.path.exists(RES):
        os.remove(RES)
    print('no prior results file on the repo - fresh start (96 runs)')


In [ ]:
import importlib.util, time, torch
import torch.nn.functional as F

spec = importlib.util.spec_from_file_location('h',
                                              'p-rick/code/p-027b-training.py')
h = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h)

def bench(dev, depth=240, steps=3):
    torch.manual_seed(0)
    m = h.StreamNet('ph', depth).to(dev)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    x = torch.randn(256, h.D, device=dev)
    y = torch.randint(0, 4, (256,), device=dev)
    for _ in range(1):   # warmup
        loss = F.cross_entropy(m(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    if dev == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(steps):
        loss = F.cross_entropy(m(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    if dev == 'cuda':
        torch.cuda.synchronize()
    return (time.time() - t0) / steps

cpu = bench('cpu')
gpu = bench('cuda') if torch.cuda.is_available() else float('inf')
DEV = 'cuda' if gpu < cpu else 'cpu'
print(f'depth-240 PH training step:  cpu {cpu*1000:6.0f} ms   '
      f'cuda {gpu*1000:6.0f} ms')
print(f'-> running the sweep on {DEV.upper()}'
      + ('' if DEV == 'cuda' else '  (tiny d=32 models: launch overhead '
         'makes CPU competitive; results identical to 1e-7)'))


## The sweep — the long cell

Each job prints one line as it finishes (`test`, `val`, `grad_max`,
`fwd`, `wall`). Expect **45–90 minutes**. Progress saves after every run:
you can interrupt, and re-running this notebook resumes exactly where it
stopped. Keep this tab open so the runtime stays alive.


In [ ]:
print(f'running the remaining sweep on {DEV} ...')
!python p-rick/code/p-027b-training.py --device={DEV}


In [ ]:
from IPython.display import Image, display
import os

f7 = 'p-rick/figures/p-027/f7-training.png'
if os.path.exists(f7):
    display(Image(f7))
else:
    print('f7-training.png not written yet - the sweep was interrupted;')
    print('re-run the notebook to finish the remaining jobs.')


In [ ]:
import json
import statistics as st
from collections import defaultdict

doc = json.load(open('p-rick/figures/p-027/training-results.json'))
runs = doc['runs']
by = defaultdict(dict)          # (task,depth,model) -> {seed: best run}
for r in runs:
    k = (r['task'], r['depth'], r['model'])
    if r['seed'] not in by[k] or r['val_acc'] > by[k][r['seed']]['val_acc']:
        by[k][r['seed']] = r

print(f"{'task':9s} {'depth':>5s} {'model':10s} {'test acc (mean+-std, 3 seeds)':>30s} "
      f"{'grad max':>9s} {'fwd growth':>11s} {'n':>2s}")
for (task, depth, model), seeds in sorted(by.items()):
    accs = [s['test_acc'] for s in seeds.values()]
    gm = [s['grad_max_ratio'] for s in seeds.values()]
    fg = [s['fwd_growth'] for s in seeds.values()]
    print(f"{task:9s} {depth:5d} {model:10s} "
          f"{st.mean(accs):11.3f} +- {st.stdev(accs) if len(accs)>1 else 0:.3f}   "
          f"{st.mean(gm):9.2f} {st.mean(fg):11.2f} {len(accs):2d}")
print()
print('complete:', doc.get('complete'), '| total runs:', len(runs))


In [ ]:
import json, os
from google.colab import files

RES = 'p-rick/figures/p-027/training-results.json'
doc = json.load(open(RES))
txt = open(RES).read()
print(f"training-results.json: {len(txt)/1000:.1f} KB, "
      f"{len(doc['runs'])} runs, complete={doc.get('complete')}")
print()
print('=' * 25 + ' COPY EVERYTHING BELOW THIS LINE ' + '=' * 25)
print(txt)
print('=' * 25 + ' COPY EVERYTHING ABOVE THIS LINE ' + '=' * 25)
print()
files.download(RES)
f7 = 'p-rick/figures/p-027/f7-training.png'
if os.path.exists(f7):
    files.download(f7)


## Then what

Paste the payload between the two marker lines back to the assistant
(or just upload the downloaded `training-results.json`). The assistant will:

1. fill the E6 numbers into the paper (Section 6 + the limitations section),
2. regenerate `figures/p-027/f7-training.png` and the reading edition,
3. rebuild the typeset PDF,
4. update the site card, README, and essay — and push.

If the JSON is too long to paste, the digest table plus a handful of
representative run lines is enough to start; the downloaded file has
everything.
